In [ ]:
import os
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, logging

# hide unset weights warning
logging.set_verbosity_error()

In [ ]:
DIRS = ["bart", "bert", "gpt2", "roberta"]
MODELNAMES = [
    "facebook/bart-large",
    "google-bert/bert-base-uncased",
    "openai-community/gpt2",
    "FacebookAI/roberta-base"
]
SPLITS = [
    "train-original",
    "train-augmented-wordnet",
    "train-augmented-lesk",
    "train-augmented-bert",
    "train-augmented-hybert",
    "train-augmented-gpt2",
    "train-augmented-hygpt2"
]

In [ ]:
os.mkdir("models")

In [ ]:
for dir, mname in zip(DIRS, MODELNAMES):
    print(f"Training {mname} models...")
    for split in SPLITS:
        # 
        # Load Objects
        # 
        model = AutoModelForSequenceClassification.from_pretrained(mname, device_map='cuda')
        tokenizer = AutoTokenizer.from_pretrained(mname, device_map='cuda')
        data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

        if dir == 'gpt2':
            tokenizer.pad_token = tokenizer.eos_token
            model.config.pad_token_id = model.config.eos_token_id
        # 
        # Load & Prepare Data
        # 
        train_dataset = load_from_disk("dataset.hf")[split]
        tokenized_train_dataset = train_dataset.map(
            lambda batch: tokenizer(batch['text']),
            batched=True,
            remove_columns=['text']
        )
        # 
        # Train
        # 
        training_arguments = TrainingArguments(
            output_dir="models",
            per_device_train_batch_size=16,
            gradient_accumulation_steps=64,
            learning_rate=0.00001,
            num_train_epochs=5,
            save_strategy='no',
            fp16=True
        )
        trainer = Trainer(
            model=model,
            args=training_arguments,
            data_collator=data_collator,
            train_dataset=tokenized_train_dataset
        )
        trainer.train()
        model.save_pretrained(f"models/{dir}" + split[split.find('-'):])
        tokenizer.save_pretrained(f"models/{dir}" + split[split.find('-'):])